# Italy Lower-Secondary School Analysis (`Scuola media`)

This notebook examines the latest official ISTAT data for `Secondaria I grado` and focuses on the structure and territorial dynamics of Italian middle school in the 2024/2025 proxy window.

Main questions:
- Do we have direct bocciatura or repeater data for lower secondary?
- What official lower-secondary indicators are currently available?
- How strong is the exam-failure signal across territories?
- Which territorial patterns matter most for understanding lower-secondary inequality?

## Methodological Note

Source used here:
- ISTAT SDMX flow `52_1044_DF_DCIS_SCUOLE_10` (`Secondaria I grado - indicatori scolastici`)

Important limitation:
- ISTAT does **not** expose a direct lower-secondary `ripetenti` flow in the currently available school indicator registry.
- For middle school, the closest official outcome signal is `EXAM = licenziati per 100 esaminati`.
- This notebook therefore uses `failure_at_exam_proxy = 100 - licenziati per 100 esaminati`.

Interpretation rule used here:
- `TIME_PERIOD = 2024` is treated as school-year proxy `2024/2025`.
- Because the pass rate is extremely high, the lower-secondary story is more about structural context than mass failure.

## Link To Upper Secondary (Bridge Check)

To keep this notebook connected to upper-secondary outcomes, we add a compact bridge check:
- lower-secondary exam-failure proxy (latest year)
- upper-secondary repeaters (latest year)
- upper-secondary first-year repeaters (latest year)

A full multi-chart linkage analysis is available in:
- `Notebooks/italy_middle_to_upper_transition_analysis.ipynb`

In [ ]:
from pathlib import Path
import pandas as pd

ROOT = Path.cwd()
if ROOT.name == 'Notebooks':
    ROOT = ROOT.parent

upper_long = pd.read_csv(ROOT / 'local_data' / 'ISTAT' / 'school_outcomes' / 'istat_repeaters_upper_secondary_long.csv')
lower_latest = pd.read_csv(ROOT / 'local_data' / 'processed' / 'istat_lower_secondary_indicators_latest.csv')

upper_long['TIME_PERIOD'] = pd.to_numeric(upper_long['TIME_PERIOD'], errors='coerce')
upper_long['OBS_VALUE'] = pd.to_numeric(upper_long['OBS_VALUE'], errors='coerce')
lower_latest['TIME_PERIOD'] = pd.to_numeric(lower_latest['TIME_PERIOD'], errors='coerce')
lower_latest['OBS_VALUE'] = pd.to_numeric(lower_latest['OBS_VALUE'], errors='coerce')

latest_year = int(max(upper_long['TIME_PERIOD'].max(), lower_latest['TIME_PERIOD'].max()))

lower_it = lower_latest[
    (lower_latest['REF_AREA'] == 'IT')
    & (lower_latest['TYPE_SCHOOL_MANAGEMENT'] == 'ALL')
    & (lower_latest['DATA_TYPE'] == 'EXAM')
]
lower_failure = 100 - float(lower_it['OBS_VALUE'].iloc[0])

base_filter = (
    (upper_long['REF_AREA'] == 'IT')
    & (upper_long['TIME_PERIOD'] == latest_year)
    & (upper_long['TYPE_SCHOOL'] == 'ALL')
    & (upper_long['SEX'] == 'T')
    & (upper_long['CITIZENSHIP'] == 'TOTAL')
    & (upper_long['TYPE_SCHOOL_MANAGEMENT'] == 'ALL')
)

upper_all = float(upper_long[base_filter & (upper_long['SCHOOL_YEAR'] == 'ALL')]['OBS_VALUE'].iloc[0])
upper_fir = float(upper_long[base_filter & (upper_long['SCHOOL_YEAR'] == 'FIR')]['OBS_VALUE'].iloc[0])

pd.DataFrame([
    {'metric': 'Lower-secondary exam failure proxy', 'value': lower_failure},
    {'metric': 'Upper-secondary repeaters (ALL years)', 'value': upper_all},
    {'metric': 'Upper-secondary repeaters (first year only)', 'value': upper_fir},
])

Modeling outputs available after running the transition notebook:
- `local_data/processed/transition_bridge_model_panel.csv`
- `local_data/processed/transition_bridge_latest_top_jump.csv`

These files are built to estimate transition-risk models with lagged lower-secondary predictors and upper-secondary outcomes.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid', context='talk')
pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 150)

ROOT = Path.cwd()
if ROOT.name == 'Notebooks':
    ROOT = ROOT.parent

LONG_PATH = ROOT / 'local_data' / 'ISTAT' / 'school_outcomes' / 'istat_lower_secondary_indicators_long.csv'
LATEST_PATH = ROOT / 'local_data' / 'processed' / 'istat_lower_secondary_indicators_latest.csv'
EXAM_PROXY_PATH = ROOT / 'local_data' / 'processed' / 'istat_lower_secondary_exam_proxy_latest.csv'
MANIFEST_PATH = ROOT / 'local_data' / 'processed' / 'istat_lower_secondary_sources_manifest.csv'

indicators = pd.read_csv(LONG_PATH)
latest = pd.read_csv(LATEST_PATH)
exam_proxy = pd.read_csv(EXAM_PROXY_PATH)
manifest = pd.read_csv(MANIFEST_PATH)

for frame in (indicators, latest, exam_proxy):
    if 'TIME_PERIOD' in frame.columns:
        frame['TIME_PERIOD'] = pd.to_numeric(frame['TIME_PERIOD'], errors='coerce')
    if 'OBS_VALUE' in frame.columns:
        frame['OBS_VALUE'] = pd.to_numeric(frame['OBS_VALUE'], errors='coerce')
    if 'failure_at_exam_proxy' in frame.columns:
        frame['failure_at_exam_proxy'] = pd.to_numeric(frame['failure_at_exam_proxy'], errors='coerce')

def classify_territory_level(code):
    code = str(code)
    if code == 'IT':
        return 'national'
    if len(code) == 3:
        return 'macroarea'
    if len(code) == 4:
        return 'region'
    if len(code) == 5:
        return 'province_like'
    return 'other'

indicators['territory_level'] = indicators['REF_AREA'].astype(str).map(classify_territory_level)
latest['territory_level'] = latest['REF_AREA'].astype(str).map(classify_territory_level)
exam_proxy['territory_level'] = exam_proxy['REF_AREA'].astype(str).map(classify_territory_level)

latest_year = int(indicators['TIME_PERIOD'].max())
latest_proxy = indicators.loc[indicators['TIME_PERIOD'] == latest_year, 'SCHOOL_YEAR_PROXY'].dropna().iloc[0]

print('Rows loaded:', len(indicators))
print('Latest period:', latest_year, '=>', latest_proxy)

In [ ]:
manifest

## Build Clean Analysis Views

For most comparisons below, we focus on:
- all school-management types together when comparing territories (`ALL`)
- regions for the main territorial plots
- provinces and similar territorial units for hotspot views

In [ ]:
national = indicators[(indicators['REF_AREA'] == 'IT') & (indicators['TYPE_SCHOOL_MANAGEMENT'] == 'ALL')].copy()
national_latest = national[national['TIME_PERIOD'] == latest_year].copy()
regional_latest = latest[(latest['territory_level'] == 'region') & (latest['TYPE_SCHOOL_MANAGEMENT'] == 'ALL')].copy()
regional_wide = regional_latest.pivot_table(index='REF_AREA_LABEL', columns='DATA_TYPE', values='OBS_VALUE')

national_latest[['DATA_TYPE', 'DATA_TYPE_LABEL', 'OBS_VALUE', 'failure_at_exam_proxy']]

## 1. National Trend of Key Lower-Secondary Indicators

Middle school is not dominated by a large failure phenomenon in the official data. The trend is more informative when we look at several indicators together: class size, foreign-student share, disability share, public-school share, exam pass rate, and median exit grade.

In [ ]:
key_codes = ['DISAB', 'ENROL', 'EXAM', 'FOR', 'LMG', 'PUB']
trend = national[national['DATA_TYPE'].isin(key_codes)].copy()
fig, axes = plt.subplots(3, 2, figsize=(16, 15), sharex=True)
axes = axes.flatten()
for ax, code in zip(axes, key_codes):
    sub = trend[trend['DATA_TYPE'] == code].sort_values('TIME_PERIOD')
    label = sub['DATA_TYPE_LABEL'].iloc[0]
    ax.plot(sub['TIME_PERIOD'], sub['OBS_VALUE'], marker='o', linewidth=2.5)
    ax.set_title(label, fontweight='bold')
    ax.set_xlabel('Year')
    ax.set_ylabel('Value')
plt.suptitle('Italy lower-secondary indicators - national trend', y=1.02, fontsize=20, fontweight='bold')
plt.tight_layout()
plt.show()

## 2. Latest National Snapshot

This table and chart summarize what the lower-secondary system looks like nationally in the latest available year.

In [ ]:
latest_small = national_latest[national_latest['DATA_TYPE'].isin(['DISAB', 'ENROL', 'EXAM', 'FEM', 'FOR', 'LMG', 'PUB'])].copy()
latest_small = latest_small.sort_values('OBS_VALUE', ascending=True)
fig, ax = plt.subplots(figsize=(11, 7))
bars = ax.barh(latest_small['DATA_TYPE_LABEL'], latest_small['OBS_VALUE'], color='#5c6bc0')
ax.set_title(f'Italy lower-secondary snapshot - {latest_proxy}', fontweight='bold')
ax.set_xlabel('Indicator value')
ax.set_ylabel('Indicator')
for bar in bars:
    width = bar.get_width()
    ax.text(width + (0.4 if width < 100 else 5000), bar.get_y() + bar.get_height()/2, f'{width:.1f}', va='center', fontsize=10)
plt.tight_layout()
plt.show()

national_latest[['DATA_TYPE', 'DATA_TYPE_LABEL', 'OBS_VALUE', 'failure_at_exam_proxy']].sort_values('DATA_TYPE')

## 3. Exam-Failure Proxy Across Territories

This is the closest official lower-secondary outcome proxy available here. It is very small everywhere, which already tells us that lower-secondary failure at the exit exam is structurally limited compared with upper secondary.

In [ ]:
exam_regions = latest[(latest['DATA_TYPE'] == 'EXAM') & (latest['TYPE_SCHOOL_MANAGEMENT'] == 'ALL') & (latest['territory_level'] == 'region')].copy()
exam_regions['failure_at_exam_proxy'] = 100 - exam_regions['OBS_VALUE']
exam_regions = exam_regions.sort_values('failure_at_exam_proxy', ascending=False)

fig, ax = plt.subplots(figsize=(12, 7))
sns.barplot(data=exam_regions, y='REF_AREA_LABEL', x='failure_at_exam_proxy', hue='REF_AREA_LABEL', palette='Reds_r', dodge=False, legend=False, ax=ax)
ax.set_title(f'Regional lower-secondary exam-failure proxy - {latest_proxy}', fontweight='bold')
ax.set_xlabel('Failure at exam proxy = 100 - licenziati per 100 esaminati')
ax.set_ylabel('Region')
plt.tight_layout()
plt.show()

exam_regions[['REF_AREA_LABEL', 'OBS_VALUE', 'failure_at_exam_proxy']].head(10)

## 4. Structural Territorial Context

Because the exam-failure proxy is extremely compressed, territorial differentiation is easier to see through two structural indicators:
- foreign-student share
- disability share per 1000 enrolled students

In [ ]:
for_regions = latest[(latest['DATA_TYPE'] == 'FOR') & (latest['TYPE_SCHOOL_MANAGEMENT'] == 'ALL') & (latest['territory_level'] == 'region')].copy()
for_regions = for_regions.sort_values('OBS_VALUE', ascending=False)

fig, ax = plt.subplots(figsize=(12, 7))
sns.barplot(data=for_regions, y='REF_AREA_LABEL', x='OBS_VALUE', hue='REF_AREA_LABEL', palette='Blues_r', dodge=False, legend=False, ax=ax)
ax.set_title(f'Regional share of foreign students in lower secondary - {latest_proxy}', fontweight='bold')
ax.set_xlabel('Foreign students per 100 students')
ax.set_ylabel('Region')
plt.tight_layout()
plt.show()

for_regions[['REF_AREA_LABEL', 'OBS_VALUE']].head(10)

In [ ]:
disab_regions = latest[(latest['DATA_TYPE'] == 'DISAB') & (latest['TYPE_SCHOOL_MANAGEMENT'] == 'ALL') & (latest['territory_level'] == 'region')].copy()
disab_regions = disab_regions.sort_values('OBS_VALUE', ascending=False)

fig, ax = plt.subplots(figsize=(12, 7))
sns.barplot(data=disab_regions, y='REF_AREA_LABEL', x='OBS_VALUE', hue='REF_AREA_LABEL', palette='Purples_r', dodge=False, legend=False, ax=ax)
ax.set_title(f'Regional disability indicator in lower secondary - {latest_proxy}', fontweight='bold')
ax.set_xlabel('Students with disabilities per 1000 students')
ax.set_ylabel('Region')
plt.tight_layout()
plt.show()

disab_regions[['REF_AREA_LABEL', 'OBS_VALUE']].head(10)

In [ ]:
context = regional_wide[['DISAB', 'FOR', 'EXAM', 'LMG']].dropna().reset_index()
context['failure_at_exam_proxy'] = 100 - context['EXAM']
fig, ax = plt.subplots(figsize=(11, 8))
sns.scatterplot(data=context, x='FOR', y='failure_at_exam_proxy', size='DISAB', hue='LMG', palette='viridis', sizes=(60, 300), ax=ax)
for row in context.itertuples():
    ax.text(row.FOR + 0.05, row.failure_at_exam_proxy + 0.002, row.REF_AREA_LABEL, fontsize=8)
ax.set_title(f'Foreign-student share and exam-failure proxy by region - {latest_proxy}', fontweight='bold')
ax.set_xlabel('Foreign students per 100 students')
ax.set_ylabel('Failure at exam proxy')
plt.tight_layout()
plt.show()

## 5. Public vs Private Management

The lower-secondary system is overwhelmingly public, but it is still useful to check whether the public/private mix changes the main indicators.

In [ ]:
management_latest = latest[(latest['REF_AREA'] == 'IT') & (latest['TYPE_SCHOOL_MANAGEMENT'].isin(['PUB', 'PRI'])) & (latest['DATA_TYPE'].isin(['ENROL', 'EXAM', 'FOR', 'DISAB', 'LMG']))].copy()
management_latest = management_latest.sort_values(['DATA_TYPE', 'TYPE_SCHOOL_MANAGEMENT'])

fig, axes = plt.subplots(3, 2, figsize=(15, 14))
axes = axes.flatten()
for ax, code in zip(axes, ['ENROL', 'EXAM', 'FOR', 'DISAB', 'LMG']):
    sub = management_latest[management_latest['DATA_TYPE'] == code].copy()
    label = sub['DATA_TYPE_LABEL'].iloc[0]
    sns.barplot(data=sub, x='TYPE_SCHOOL_MANAGEMENT_LABEL', y='OBS_VALUE', hue='TYPE_SCHOOL_MANAGEMENT_LABEL', dodge=False, legend=False, palette='Set2', ax=ax)
    ax.set_title(label, fontweight='bold')
    ax.set_xlabel('Management type')
    ax.set_ylabel('Value')
axes[-1].axis('off')
plt.suptitle(f'Italy lower-secondary indicators by management type - {latest_proxy}', y=1.02, fontsize=20, fontweight='bold')
plt.tight_layout()
plt.show()

management_latest[['DATA_TYPE', 'DATA_TYPE_LABEL', 'TYPE_SCHOOL_MANAGEMENT_LABEL', 'OBS_VALUE']].sort_values(['DATA_TYPE', 'TYPE_SCHOOL_MANAGEMENT_LABEL'])

## 6. Automated Findings

This final cell calculates the main conclusions directly from the currently refreshed data.

In [ ]:
nat_exam = float(national_latest.loc[national_latest['DATA_TYPE'] == 'EXAM', 'OBS_VALUE'].iloc[0])
nat_failure = 100 - nat_exam
nat_for = float(national_latest.loc[national_latest['DATA_TYPE'] == 'FOR', 'OBS_VALUE'].iloc[0])
nat_disab = float(national_latest.loc[national_latest['DATA_TYPE'] == 'DISAB', 'OBS_VALUE'].iloc[0])
nat_enrol = float(national_latest.loc[national_latest['DATA_TYPE'] == 'ENROL', 'OBS_VALUE'].iloc[0])
nat_lmg = float(national_latest.loc[national_latest['DATA_TYPE'] == 'LMG', 'OBS_VALUE'].iloc[0])
nat_pub = float(national_latest.loc[national_latest['DATA_TYPE'] == 'PUB', 'OBS_VALUE'].iloc[0])

worst_exam_region = exam_regions.iloc[0]
best_exam_region = exam_regions.iloc[-1]
top_foreign_region = for_regions.iloc[0]
top_disab_region = disab_regions.iloc[0]

trend_prev = national[(national['TIME_PERIOD'] == latest_year - 1) & (national['DATA_TYPE'].isin(['EXAM', 'FOR', 'DISAB', 'ENROL', 'LMG', 'PUB']))][['DATA_TYPE', 'OBS_VALUE']].rename(columns={'OBS_VALUE': 'prev'})
trend_curr = national_latest[national_latest['DATA_TYPE'].isin(['EXAM', 'FOR', 'DISAB', 'ENROL', 'LMG', 'PUB'])][['DATA_TYPE', 'DATA_TYPE_LABEL', 'OBS_VALUE']].rename(columns={'OBS_VALUE': 'curr'})
changes = trend_curr.merge(trend_prev, on='DATA_TYPE', how='left')
changes['change'] = changes['curr'] - changes['prev']

print(f'Latest proxy year: {latest_proxy}')
print()
print('Key findings')
print(f'- National exam pass indicator: {nat_exam:.1f} licenziati per 100 esaminati')
print(f'- National exam-failure proxy: {nat_failure:.1f}')
print(f'- National foreign-student share: {nat_for:.1f} per 100 students')
print(f'- National disability indicator: {nat_disab:.1f} per 1000 students')
print(f'- National class size indicator: {nat_enrol:.1f} students per class')
print(f'- National median lower-secondary grade: {nat_lmg:.1f}')
print(f'- Public-school share: {nat_pub:.1f}')
print()
print('Territorial signals')
worst_exam_name = worst_exam_region['REF_AREA_LABEL']
worst_exam_value = float(worst_exam_region['failure_at_exam_proxy'])
best_exam_name = best_exam_region['REF_AREA_LABEL']
best_exam_value = float(best_exam_region['failure_at_exam_proxy'])
top_foreign_name = top_foreign_region['REF_AREA_LABEL']
top_foreign_value = float(top_foreign_region['OBS_VALUE'])
top_disab_name = top_disab_region['REF_AREA_LABEL']
top_disab_value = float(top_disab_region['OBS_VALUE'])
print(f'- Highest exam-failure proxy region: {worst_exam_name} ({worst_exam_value:.2f})')
print(f'- Lowest exam-failure proxy region: {best_exam_name} ({best_exam_value:.2f})')
print(f'- Highest regional foreign-student share: {top_foreign_name} ({top_foreign_value:.1f})')
print(f'- Highest regional disability indicator: {top_disab_name} ({top_disab_value:.1f})')
print()
print('Year-on-year movement vs previous year')
for row in changes.sort_values('DATA_TYPE').itertuples():
    direction = 'down' if row.change < 0 else 'up'
    print(f'- {row.DATA_TYPE_LABEL}: {row.curr:.1f} ({direction} {abs(row.change):.1f})')

## Interpretation

The lower-secondary case differs sharply from upper secondary:
- middle school does not show a large official failure signal in the available ISTAT outcome indicator
- the exam pass rate is near-universal almost everywhere
- territorial differentiation is more visible in system composition and student profile than in exit failure itself
- this means lower secondary is better read as a filtering and preparation stage whose strongest inequalities may emerge later in upper secondary rather than at the middle-school exit exam itself

## NEET Outcomes: Post-Secondary Labour-Market Engagement

Lower secondary completion feeds into NEET risk. Youth who don't continue to upper secondary and don't secure employment constitute NEET population (Not in Employment, Education, or Training). Use ANPAL replacement datasets to assess outcomes.

In [ ]:
# Load NEET replacement data (substitute for missing ANPAL Garanzia Giovani)
processed_dir = Path("../local_data/processed")

anpal_neet = pd.read_csv(processed_dir / "anpal_replacement_neet_annual.csv")
print(f"Annual NEET rate by year: {len(anpal_neet)} years")
print(anpal_neet)

# Visualization: NEET rate trend
fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(anpal_neet['anno'], anpal_neet['neet_rate_pct'], marker='o', linewidth=2.5, color='#e74c3c')
ax.set_xlabel('Year', fontsize=11)
ax.set_ylabel('NEET Rate (%)', fontsize=11)
ax.set_title('Youth NEET Rate: Outcome for Non-Transition Cases', fontsize=12, fontweight='bold')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Analysis: Use to contextualize lower secondary outcomes
print("\n📌 Interpretation:")
print("   - Rising NEET rate → larger pool of youth without education/employment")
print("   - Lower secondary completers who don't transition to upper secondary at higher risk")
print("   - Policy goal: Maximize transition rate to reduce NEET flow from lower secondary cohorts")
